# Device Management and GPU Usage

Hardware mistakes in PyTorch are usually small, local, and expensive. Accidentally moving tensors back to CPU, choosing a batch size that underuses the GPU, or calling synchronizing operations too often can wipe out the speedup you expected.

This notebook stays close to native PyTorch and focuses on three habits:

1. Choose and inspect the device explicitly.
2. Move tensors in one direction, at well-defined points.
3. Measure throughput before and after changing batch size or precision.


In [1]:
import time

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

def choose_device(preferred_index=0):
    if torch.cuda.is_available():
        count = torch.cuda.device_count()
        idx = min(preferred_index, count - 1)
        return torch.device(f'cuda:{idx}')
    return torch.device('cpu')

device = choose_device()
print(f'Using device: {device}')
if torch.cuda.is_available():
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        print(f'cuda:{idx} | {props.name} | {props.total_memory / 1024**3:.1f} GB')
else:
    print('No CUDA device found. The notebook still demonstrates the code structure, but GPU-specific differences will not appear.')


Using device: cpu
No CUDA device found. The notebook still demonstrates the code structure, but GPU-specific differences will not appear.


In [2]:
n_samples = 8192
n_features = 256
n_classes = 4

X = torch.randn(n_samples, n_features)
W = torch.randn(n_features, n_classes)
logits = X @ W + 0.2 * torch.randn(n_samples, n_classes)
y = logits.argmax(dim=1)

dataset = TensorDataset(X, y)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.net(x)


## Batch-Size Sweep

Throughput almost always changes with batch size. On a GPU, very small batches often underutilize the hardware. On a CPU, the relationship can be different, so it is still worth measuring.


In [3]:
def benchmark_epoch(batch_size, use_autocast=False):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=torch.cuda.is_available())
    model = MLP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    amp_enabled = use_autocast and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)

    start = time.perf_counter()
    total = 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            pred = model(xb)
            loss = criterion(pred, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += xb.size(0)

    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return {
        'batch_size': batch_size,
        'autocast': amp_enabled,
        'seconds': elapsed,
        'samples_per_second': total / elapsed,
    }

rows = []
for batch_size in [32, 64, 128, 256, 512]:
    rows.append(benchmark_epoch(batch_size, use_autocast=False))
    rows.append(benchmark_epoch(batch_size, use_autocast=True))

batch_results = pd.DataFrame(rows)
batch_results


,batch_size,autocast,seconds,samples_per_second
0,32,False,0.360091,22749.775911
1,32,False,0.325155,25194.125320
2,64,False,0.189037,43335.403038
3,64,False,0.183435,44658.759460
4,128,False,0.118633,69053.104382
5,128,False,0.119473,68567.937042
6,256,False,0.078957,103752.072421
7,256,False,0.079366,103218.599626
8,512,False,0.059190,138402.828071
9,512,False,0.057080,143518.706951


## A Bad Transfer Pattern vs a Better One

The next pair of loops does the same optimization work, but the first version makes two common mistakes:

- converting model outputs back to CPU every batch for logging
- forcing synchronization with `.item()` more often than necessary


In [ ]:
def bad_loop(batch_size=256):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, pin_memory=torch.cuda.is_available())
    model = MLP().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=5e-2)
    criterion = nn.CrossEntropyLoss()
    losses = []
    start = time.perf_counter()

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        _ = pred.detach().cpu().numpy()

    if device.type == 'cuda':
        torch.cuda.synchronize()
    return time.perf_counter() - start, sum(losses) / len(losses)


def better_loop(batch_size=256):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, pin_memory=torch.cuda.is_available())
    model = MLP().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=5e-2)
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    batches = 0
    start = time.perf_counter()

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

        total_loss += float(loss.detach())
        batches += 1

    if device.type == 'cuda':
        torch.cuda.synchronize()
    return time.perf_counter() - start, total_loss / batches

bad_seconds, bad_loss = bad_loop()
good_seconds, good_loss = better_loop()
pd.DataFrame([
    {'loop': 'bad', 'seconds': bad_seconds, 'mean_loss': bad_loss},
    {'loop': 'better', 'seconds': good_seconds, 'mean_loss': good_loss},
])


In [ ]:
if device.type == 'cuda':
    allocated = torch.cuda.memory_allocated(device) / 1024**2
    reserved = torch.cuda.memory_reserved(device) / 1024**2
    print(f'Memory allocated: {allocated:.1f} MiB')
    print(f'Memory reserved:  {reserved:.1f} MiB')
else:
    print('CUDA memory statistics are only available when running on a GPU.')


**Device management checklist**

1. Pick the device explicitly rather than relying on hidden defaults.
2. Move model and tensors once per batch, not back and forth inside helper functions.
3. Sweep batch size before claiming a GPU is slow.
4. Use mixed precision only after confirming your device supports it and your loss stays stable.
